# Sprint 3 — MegaMarket: ¿qué referencias nuevas lanzamos con el presupuesto que tenemos?

Jacobo Argote, Andrés Cardona, Miguel Valencia y Juan José Gil

##**CÓDIGO GENERADO CON CLAUDE**

## De dónde venimos (Sprint 2) y qué agregamos en este sprint

En el Sprint 2 construimos el **sistema experto de portafolio**: siete
criterios (C1 a C7) que un producto candidato debe cumplir *todos a la
vez* para ser declarado **VIABLE**. Es un chequeo tipo "todas las luces
en verde" — no compara candidatos entre sí, y no dice nada sobre
*cuántos* de ellos se pueden lanzar al tiempo.

El problema real que se nos queda sin resolver: el gerente general
amplió la lista a **8 candidatos que ya pasaron los 7 criterios**, pero
MegaMarket no tiene presupuesto ni espacio de bodega para lanzar los 8
en el mismo semestre. Hay que **elegir un subconjunto** que aproveche
mejor los recursos disponibles — y con 8 candidatos ya hay 2⁸ = 256
subconjuntos posibles, así que a mano no es trivial.

Este es exactamente el tipo de problema que un **algoritmo genético**
resuelve bien: existen muchas soluciones posibles, una solución se
puede representar como cromosoma, se puede medir su calidad con un
*fitness*, y hay restricciones que respetar (presupuesto, espacio). Es
la versión de "mochila" (*knapsack*) del algoritmo genético que vimos
en el cuaderno del profesor — en vez de escoger el orden de una ruta,
escogemos *cuáles* productos entran.

> **Nota:** los montos de costo, espacio y utilidad son ficticios,
> igual que los tiempos y costos del ejemplo del profesor. El objetivo
> es mostrar la lógica de selección bajo restricciones, no representar
> cifras reales de MegaMarket.

## Parte 0 — Candidatos que ya pasaron el filtro del Sprint 2

Los 8 productos de abajo ya fueron evaluados por
`motor_inferencia_portafolio` del Sprint 2 y los 8 salieron **VIABLE**
(cumplen C1 a C7). Acá no se repite esa evaluación — se agregan los
tres datos numéricos que la decisión de *presupuesto* necesita y que
el sistema experto nunca calculó, porque su lógica era binaria (Sí/No),
no numérica:

- **costo de lanzamiento**: registro sanitario, empaque, mercadeo inicial.
- **espacio de bodega**: posiciones que ocuparía la referencia nueva.
- **utilidad esperada anual**: margen × ventas proyectadas para el primer año.

In [ ]:
# =============================================================================
# 1. CANDIDATOS — ya viables según el Sprint 2, con sus datos de negocio
# =============================================================================
CANDIDATOS = {
    "Higo Deshidratado Riovalle":     {"costo_lanzamiento": 18_000_000, "espacio_bodega": 6,  "utilidad_esperada_anual": 32_000_000},
    "Mix de Frutos Secos Riovalle":   {"costo_lanzamiento": 25_000_000, "espacio_bodega": 8,  "utilidad_esperada_anual": 48_000_000},
    "Barra Energética Riovalle":      {"costo_lanzamiento": 15_000_000, "espacio_bodega": 4,  "utilidad_esperada_anual": 22_000_000},
    "Granola Riovalle":               {"costo_lanzamiento": 20_000_000, "espacio_bodega": 7,  "utilidad_esperada_anual": 30_000_000},
    "Jugo Natural Riovalle":          {"costo_lanzamiento": 30_000_000, "espacio_bodega": 10, "utilidad_esperada_anual": 52_000_000},
    "Snack de Plátano Riovalle":      {"costo_lanzamiento": 12_000_000, "espacio_bodega": 5,  "utilidad_esperada_anual": 17_000_000},
    "Compota Riovalle":               {"costo_lanzamiento": 10_000_000, "espacio_bodega": 3,  "utilidad_esperada_anual": 14_000_000},
    "Café Molido Riovalle":           {"costo_lanzamiento": 22_000_000, "espacio_bodega": 6,  "utilidad_esperada_anual": 38_000_000},
}

# El orden de esta lista define la posición de cada gen en el cromosoma.
NOMBRES = list(CANDIDATOS.keys())
N_GENES = len(NOMBRES)

print(f"Candidatos viables a evaluar: {N_GENES}")
for nombre in NOMBRES:
    print(" •", nombre)


## Parte 1 — Restricciones: presupuesto y espacio de bodega

Estas son las políticas de la compañía para este semestre — el
equivalente a los umbrales (`dias_sin_entrega`, `flete_sobre_factura`)
que ya vieron en la base de conocimiento del Sprint 2.

In [ ]:
# =============================================================================
# 2. RESTRICCIONES DE LA DECISIÓN
# =============================================================================
PRESUPUESTO_TOTAL = 70_000_000        # COP disponibles para lanzamientos este semestre
ESPACIO_BODEGA_TOTAL = 25             # posiciones de bodega disponibles para SKU nuevas

costo_si_se_lanzan_todos = sum(c["costo_lanzamiento"] for c in CANDIDATOS.values())
espacio_si_se_lanzan_todos = sum(c["espacio_bodega"] for c in CANDIDATOS.values())

print(f"Si se lanzaran los {N_GENES} candidatos a la vez:")
print(f" Costo total:   ${costo_si_se_lanzan_todos:,.0f} (presupuesto disponible: ${PRESUPUESTO_TOTAL:,.0f})")
print(f" Espacio total: {espacio_si_se_lanzan_todos} posiciones (disponibles: {ESPACIO_BODEGA_TOTAL})")
print(" -> ninguno de los dos recursos alcanza. Toca elegir un subconjunto.")


In [ ]:
# @title
# CÓDIGO DE APOYO VISUAL (oculto)
# Ubica cada candidato según su costo y su utilidad esperada; el tamaño
# de la burbuja representa el espacio de bodega que ocuparía.

import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(8.8, 5.6))
for nombre, datos in CANDIDATOS.items():
    ax.scatter(
        datos["costo_lanzamiento"] / 1_000_000,
        datos["utilidad_esperada_anual"] / 1_000_000,
        s=datos["espacio_bodega"] * 40,
        color="#2878B5", edgecolor="white", linewidth=1.3, alpha=0.85, zorder=3,
    )
    ax.annotate(
        nombre,
        (datos["costo_lanzamiento"] / 1_000_000, datos["utilidad_esperada_anual"] / 1_000_000),
        xytext=(7, 6), textcoords="offset points", fontsize=8.5,
    )

ax.set_xlabel("Costo de lanzamiento (millones COP)")
ax.set_ylabel("Utilidad esperada anual (millones COP)")
ax.set_title("Candidatos viables del Sprint 2 (tamaño = espacio de bodega)")
plt.tight_layout()
plt.show()


## Parte 2 — Del problema al cromosoma

A diferencia del ejemplo del profesor (donde el cromosoma era una ruta
de 4 nodos en orden), acá una solución es: **¿cuáles de los 8
candidatos se lanzan?** Eso se representa con una cadena de 8 genes
binarios, uno por candidato, en el mismo orden que `NOMBRES`:

```
(1, 0, 1, 1, 0, 0, 0, 1)
```

El gen en la posición *i* vale `1` si el candidato `NOMBRES[i]` se
lanza, y `0` si no. Es el mismo tipo de cromosoma binario que aparece
en la fila de "selección de variables" de la tabla de aplicaciones del
cuaderno del profesor — acá la "variable" que se activa o se apaga es
un producto completo.

> **Idea clave (la misma del cuaderno del profe):** el cromosoma no
> tiene que representar un camino. Cualquier lista de decisiones
> independientes (lanzar o no lanzar cada producto) puede codificarse
> como una cadena de bits.

In [ ]:
# =============================================================================
# 3. FUNCIONES BÁSICAS SOBRE UN INDIVIDUO (portafolio candidato)
# =============================================================================
def costo_total(individuo):
    """Suma el costo de lanzamiento de los productos incluidos (gen = 1)."""
    return sum(CANDIDATOS[NOMBRES[i]]["costo_lanzamiento"] for i in range(N_GENES) if individuo[i] == 1)


def espacio_total(individuo):
    """Suma el espacio de bodega de los productos incluidos."""
    return sum(CANDIDATOS[NOMBRES[i]]["espacio_bodega"] for i in range(N_GENES) if individuo[i] == 1)


def utilidad_total(individuo):
    """Suma la utilidad esperada de los productos incluidos."""
    return sum(CANDIDATOS[NOMBRES[i]]["utilidad_esperada_anual"] for i in range(N_GENES) if individuo[i] == 1)


def es_portafolio_valido(individuo):
    """Un portafolio es válido si respeta el presupuesto Y el espacio a la vez."""
    return costo_total(individuo) <= PRESUPUESTO_TOTAL and espacio_total(individuo) <= ESPACIO_BODEGA_TOTAL


def nombre_incluidos(individuo):
    """Lista los nombres de los productos incluidos en este individuo."""
    return [NOMBRES[i] for i in range(N_GENES) if individuo[i] == 1]


## Parte 3 — Reparación: qué pasa cuando el portafolio no cabe

En el ejemplo del profesor, el cruce podía producir una ruta imposible
(una conexión que no existe), y la reparación cambiaba solo la parte
incompatible. Acá el equivalente es: el cruce o la mutación pueden
producir un portafolio que se pasa del presupuesto o del espacio.

**Regla de reparación:** mientras el portafolio no sea válido, se
retira el producto incluido con la **peor relación utilidad/costo**
(el que peor retorno da por cada peso invertido), uno a la vez.
Retirar un producto libera presupuesto y espacio *simultáneamente*,
así que basta con mirar un solo criterio para decidir cuál sacar.

> **Idea clave (la misma del cuaderno del profe):** en otros problemas
> se puede penalizar en el *fitness* en vez de reparar. Acá se eligió
> reparación porque, igual que con las rutas, es fácil arreglar
> exactamente la parte que rompe la restricción sin descartar todo el
> individuo.

In [ ]:
# =============================================================================
# 4. REPARACIÓN
# =============================================================================
def reparar_portafolio(individuo):
    """Retira, uno a uno, el producto incluido con peor utilidad por peso
    invertido, hasta que el portafolio vuelva a caber en presupuesto y
    espacio a la vez."""
    individuo = list(individuo)

    while not es_portafolio_valido(individuo):
        incluidos = [i for i in range(N_GENES) if individuo[i] == 1]

        peor = min(
            incluidos,
            key=lambda i: CANDIDATOS[NOMBRES[i]]["utilidad_esperada_anual"]
                          / CANDIDATOS[NOMBRES[i]]["costo_lanzamiento"],
        )
        individuo[peor] = 0

    return tuple(individuo)


# Ejemplo: "lanzar los 8" es un candidato inválido (se calculó en la Parte 1).
candidato_demo = tuple([1] * N_GENES)
print("¿Lanzar los 8 candidatos es válido?", es_portafolio_valido(candidato_demo))

reparado_demo = reparar_portafolio(candidato_demo)
print("Después de reparar, quedan:", nombre_incluidos(reparado_demo))
print(f"Costo: ${costo_total(reparado_demo):,.0f} | Espacio: {espacio_total(reparado_demo)}")


## Parte 4 — Fitness, población y selección

El *fitness* de un portafolio es simplemente su **utilidad esperada
total**: mientras más alta, mejor. Se le suma 1 para que ningún
portafolio (ni siquiera uno vacío) tenga fitness cero — eso evita un
error técnico en la selección por ruleta, que necesita pesos positivos.

La selección sigue siendo **por ruleta**, igual que con las rutas: más
utilidad, más probabilidad de ser elegido como padre, pero sin
eliminar del todo la posibilidad de que un portafolio más modesto se
reproduzca.

In [ ]:
# =============================================================================
# 5. FITNESS, POBLACIÓN Y SELECCIÓN
# =============================================================================
import random

SEMILLA = 42
random.seed(SEMILLA)


def calcular_fitness(individuo):
    """La utilidad total es el fitness. El +1 evita pesos en cero."""
    return utilidad_total(individuo) + 1


def crear_individuo():
    """Cada gen se decide al azar (lanzar o no ese producto); si el
    resultado no cabe en presupuesto/espacio, se repara."""
    candidato = tuple(random.choice([0, 1]) for _ in range(N_GENES))
    return reparar_portafolio(candidato)


def crear_poblacion(tamano):
    """Crea una lista con la cantidad solicitada de portafolios (todos válidos)."""
    return [crear_individuo() for _ in range(tamano)]


def seleccionar_por_ruleta(poblacion):
    """Selecciona un individuo con probabilidad proporcional a su fitness."""
    pesos = [calcular_fitness(individuo) for individuo in poblacion]
    return random.choices(poblacion, weights=pesos, k=1)[0]


## Parte 5 — Cruce y mutación

**Cruce de un punto:** se elige una posición de corte al azar entre
los 8 genes; el hijo toma los genes de un padre hasta el corte, y los
del otro padre de ahí en adelante. Es la versión para cadenas de bits
del cruce que vieron con las rutas — ahí se tomaba "clasificación de un
padre, distribución del otro"; acá se toma "los primeros genes de un
padre, los últimos del otro". El resultado se repara si hace falta.

**Mutación:** con una probabilidad baja, se voltea un gen al azar
(incluir un producto que estaba fuera, o sacar uno que estaba dentro).
También se repara si el resultado ya no cabe.

In [ ]:
# =============================================================================
# 6. CRUCE Y MUTACIÓN
# =============================================================================
def cruzar(padre_1, padre_2):
    """Cruce de un punto sobre la cadena de bits; repara si es necesario."""
    punto_corte = random.randint(1, N_GENES - 1)
    candidato = padre_1[:punto_corte] + padre_2[punto_corte:]
    hijo = reparar_portafolio(candidato)

    assert es_portafolio_valido(hijo)
    return hijo


def mutar(individuo, probabilidad=0.10):
    """Voltea un gen al azar (incluir <-> excluir un producto) y repara."""
    if random.random() >= probabilidad:
        return individuo

    posicion = random.randrange(N_GENES)
    mutado = list(individuo)
    mutado[posicion] = 1 - mutado[posicion]
    mutado = reparar_portafolio(mutado)

    assert es_portafolio_valido(mutado)
    return mutado


## Parte 6 — Integración del algoritmo genético

Mismo ciclo del cuaderno del profesor, con "ruta" cambiado por
"portafolio":

1. evaluar la población;
2. conservar los mejores individuos mediante elitismo;
3. seleccionar padres;
4. crear hijos mediante cruce y reparación;
5. aplicar mutación;
6. repetir durante varias generaciones.

In [ ]:
# =============================================================================
# 7. ALGORITMO GENÉTICO — PORTAFOLIO
# =============================================================================
import pandas as pd

def algoritmo_genetico_portafolio(
    poblacion_inicial,
    generaciones=15,
    probabilidad_mutacion=0.10,
    elitismo=1,
):
    """Ejecuta el algoritmo genético sobre portafolios y registra su evolución."""
    poblacion = list(poblacion_inicial)
    tamano_poblacion = len(poblacion)
    historial = []

    for generacion in range(generaciones + 1):

        # 1. EVALUACIÓN
        poblacion.sort(key=calcular_fitness, reverse=True)

        historial.append({
            "Generación": generacion,
            "Mejor utilidad": utilidad_total(poblacion[0]),
            "Utilidad promedio": sum(utilidad_total(ind) for ind in poblacion) / tamano_poblacion,
            "Diversidad": len(set(poblacion)),
        })

        if generacion == generaciones:
            break

        # 2. ELITISMO
        nueva_poblacion = poblacion[:elitismo]

        # 3. REPRODUCCIÓN
        while len(nueva_poblacion) < tamano_poblacion:
            padre_1 = seleccionar_por_ruleta(poblacion)
            padre_2 = seleccionar_por_ruleta(poblacion)

            hijo = cruzar(padre_1, padre_2)
            hijo = mutar(hijo, probabilidad_mutacion)

            nueva_poblacion.append(hijo)

        poblacion = nueva_poblacion

    mejor_portafolio = poblacion[0]
    return mejor_portafolio, pd.DataFrame(historial)


### Experimento reproducible

Se reinicia la semilla para que el resultado sea el mismo cada vez que
se corra el notebook, igual que hicieron en el cuaderno del profesor.

In [ ]:
# =============================================================================
# 8. EJECUCIÓN
# =============================================================================
random.seed(SEMILLA)
POBLACION_INICIAL = crear_poblacion(10)

mejor_portafolio, historial = algoritmo_genetico_portafolio(
    poblacion_inicial=POBLACION_INICIAL,
    generaciones=15,
    probabilidad_mutacion=0.10,
    elitismo=1,
)

print("Resultado del algoritmo genético")
print(" Productos:", nombre_incluidos(mejor_portafolio))
print(f" Costo total:    ${costo_total(mejor_portafolio):,.0f}")
print(f" Espacio usado:  {espacio_total(mejor_portafolio)} de {ESPACIO_BODEGA_TOTAL}")
print(f" Utilidad anual: ${utilidad_total(mejor_portafolio):,.0f}")


In [ ]:
# @title
# CÓDIGO DE APOYO VISUAL (oculto)
# Panel izquierdo: evolución de la utilidad. Panel derecho: diversidad.

fig, ejes = plt.subplots(1, 2, figsize=(11.2, 4.6), gridspec_kw={"width_ratios": [1.7, 1]})
ax = ejes[0]
ax.plot(historial["Generación"], historial["Mejor utilidad"] / 1_000_000,
        marker="o", linewidth=2.2, color="#2878B5", label="Mejor utilidad")
ax.plot(historial["Generación"], historial["Utilidad promedio"] / 1_000_000,
        marker="s", linewidth=1.8, color="#E3A018", label="Utilidad promedio")
ax.set_xlabel("Generación")
ax.set_ylabel("Utilidad anual (millones COP)")
ax.set_title("Calidad de la población")
ax.set_xticks(historial["Generación"])
ax.grid(alpha=0.25)
ax.legend(loc="lower right")

ax_div = ejes[1]
ax_div.plot(historial["Generación"], historial["Diversidad"],
            marker="D", linewidth=2.0, color="#7A5AF8")
ax_div.set_xlabel("Generación")
ax_div.set_ylabel("Portafolios distintos")
ax_div.set_title("Diversidad")
ax_div.set_xticks(historial["Generación"])
ax_div.grid(alpha=0.25)

fig.suptitle("Evolución del algoritmo genético de portafolio", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Parte 7 — Validación con fuerza bruta

Con 8 candidatos hay 2⁸ = 256 combinaciones posibles — pocas para
enumerarlas todas y comprobar si el algoritmo genético encontró
realmente la mejor. Igual que en el cuaderno del profesor, esta
búsqueda exhaustiva es solo un control; no hace parte del algoritmo
genético.

In [ ]:
# @title
# La búsqueda exhaustiva comprueba el resultado; no forma parte del AG.
import itertools

mejor_fuerza_bruta, mejor_utilidad_fb = None, -1
for combinacion in itertools.product([0, 1], repeat=N_GENES):
    if es_portafolio_valido(combinacion) and utilidad_total(combinacion) > mejor_utilidad_fb:
        mejor_fuerza_bruta = combinacion
        mejor_utilidad_fb = utilidad_total(combinacion)

resumen = pd.DataFrame([{
    "Método": "Algoritmo genético",
    "Productos": ", ".join(nombre_incluidos(mejor_portafolio)),
    "Costo (COP)": costo_total(mejor_portafolio),
    "Espacio usado": espacio_total(mejor_portafolio),
    "Utilidad anual (COP)": utilidad_total(mejor_portafolio),
}, {
    "Método": "Fuerza bruta (control)",
    "Productos": ", ".join(nombre_incluidos(mejor_fuerza_bruta)),
    "Costo (COP)": costo_total(mejor_fuerza_bruta),
    "Espacio usado": espacio_total(mejor_fuerza_bruta),
    "Utilidad anual (COP)": mejor_utilidad_fb,
}])

display(
    resumen.style.format({
        "Costo (COP)": "${:,.0f}",
        "Utilidad anual (COP)": "${:,.0f}",
    }).hide(axis="index")
)

print("¿Coinciden en la utilidad óptima?", utilidad_total(mejor_portafolio) == mejor_utilidad_fb)


## Parte 8 — Interpretación del resultado

El algoritmo genético llega a **\$122.000.000** de utilidad anual con
**Higo Deshidratado, Jugo Natural y Café Molido Riovalle** — usando
exactamente los \$70.000.000 de presupuesto y 22 de las 25 posiciones
de bodega.

La fuerza bruta confirma que \$122.000.000 es la utilidad máxima
alcanzable, pero llega a ella con una combinación **distinta**:
**Mix de Frutos Secos, Barra Energética y Jugo Natural**. Ambas
combinaciones cuestan exactamente \$70.000.000 y usan 22 posiciones de
bodega — hay **más de un portafolio óptimo**, y el AG solo tenía que
encontrar uno de ellos, no necesariamente el mismo que la fuerza
bruta.

> **Lectura de la gráfica:** en el panel izquierdo, la mejor utilidad
> llega a su valor máximo desde la generación 1 y se mantiene ahí
> gracias al elitismo; la utilidad promedio sigue oscilando porque el
> cruce y la mutación se lo siguen inventando. El panel derecho muestra
> que la diversidad cae a 1 portafolio distinto (generaciones 6 a 10)
> y luego vuelve a subir — la mutación sigue explorando aunque ya se
> encontró el óptimo, tal como se esperaría de una probabilidad de
> mutación de 10% que nunca se apaga.